# The ILP Never Divides: A Failure-First Campaign on a Public Cell-Tracking Pipeline

*What 199 sparsely annotated movies, 65 submissions and an exact CPU replay taught us about where the score was
actually lost.*

**Team:** Yudai Yamauchi, Tsukasa Miyaji &nbsp;·&nbsp; **Final rank:** TBD &nbsp;·&nbsp; **Private LB:** TBD
&nbsp;·&nbsp; **Best confirmed Public LB:** 0.964

Code, tests and the submitted notebook: <https://github.com/YamauchiYudai/biohub-cell-tracking-solution>

The code cells in this notebook need no competition data. They redraw the figures from our recorded numbers and
reproduce the core ideas on small synthetic examples.

## Executive summary

Our best confirmed public score is **0.964**. We did not get there with a new detector or a new association
model. We started from the strongest public pipeline we could find (pretrained TemporalUNet3D detector,
association Transformer, ILP and rule-based post-processing, Public 0.953) and added **+0.011** with three
components, each aimed at a failure we had measured first:

- **Learned division recovery (+0.010).** With its cost settings the ILP can never output a division: a second
  daughter edge always costs more than it gains. Every division therefore comes from a rule-based post-processing
  stage, and a deliberate "divisions off" submission showed that stage is worth **0.040** on the hidden embryos. A
  small 3-frame 3D CNN (T3) now adds gated forks and removes clearly wrong rule-based forks.
- **Jump-aware relinking (b1c).** Most audited movies of one embryo contain bit-identical duplicated frames. Across such a
  frame the motion prior collapses to zero, and on the next step whole neighbourhoods are linked to the wrong cells.
  A global-translation prior applied *only* on those transitions recovers the links.
- **Jump-aware smoothing (J2).** The final coordinate smoothing stops at duplicated frames and compensates the
  jumps instead of averaging two alignments (+0.0035 on our replay; the three-decimal public score stayed 0.964).

The most useful modelling insight was negative. Seven association variants (hard negatives, rank priors, dense
matching, a Trackastra-style 5-frame model, a MeMOTR-style memory model, ...) scored 0.870-0.946 against a 0.946
control. Attributing every missed edge explained why: **45 %** of lost edges were missing from the candidate pairs
the ILP saw, and in the final pipeline **63 %** of the remaining misses are detection or localisation errors that no
linker can fix.

The most useful process insight: the public board cannot resolve changes below about 0.003 (one division on the
hidden set), and every training movie is in-sample for the public weights. So every decision was made on an
**exact CPU replay of the submitted notebook's own code** on 88 annotated movies, per embryo, with pre-registered
gates. The board was only a sanity check.

## 1. Problem frame: one metric, two very different terms

Each movie is a 3D + time light-sheet recording of a zebrafish embryo, about 100 frames at 1.625 x 0.40625 x
0.40625 um voxels. The 199 training movies come from **two embryos** (44b6: 71 movies, 6bba: 128). Ground truth
is **sparse**: only some nuclei are annotated, and an edge or a division on an unannotated cell is neither
rewarded nor penalised.

```
score = adjusted edge Jaccard + 0.1 x division Jaccard
adjusted edge Jaccard (per movie) = edge Jaccard x (1 - 0.1 x (n_pred - n_est) / n_est)
```

Three properties of this metric shaped everything we did:

1. **The division term is small in weight but not in effect.** Turning rule-based divisions off moved our public
   score from 0.954 to 0.914. Divisions were worth 0.040, a bigger lever than any linking change we tried.
2. **The division term rests on very few events.** Across all 199 training movies there are only 151 annotated
   divisions (in 87 movies). On 40 training movies, 565 of the 598 forks we predicted sat on unannotated cells and
   were not scored at all. On the public set, one division moves the score by roughly 0.003.
3. **The node factor has no upper bound.** Predicting fewer nodes than the organisers' estimate *raises* the
   adjusted edge Jaccard. Every stage that adds nodes (re-admission, gap filling, a shorter minimum track
   length) has to pay for them.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree

BLUE, GREY, INK, MUTED, GRID = "#2a78d6", "#b4b2aa", "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": GRID, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
})
VOXEL_UM = np.array([1.625, 0.40625, 0.40625])

### Division arithmetic: why we add conservatively and almost never delete

Adding one fork either turns a missed division into a true positive or creates a false positive. With
`J = TP / (TP + FP + FN)`, adding forks with precision `p` is break-even at `p* = J / (1 + J)`. Deleting a fork
is the mirror image, except that most forks sit on unannotated cells, where the metric cannot tell us whether the
division was real.

In [ ]:
def division_j(tp, fp, fn):
    return tp / (tp + fp + fn)

# S2 on our 88-movie replay: 44 TP / 35 FP / 107 FN
tp, fp, fn = 44, 35, 107
j = division_j(tp, fp, fn)
print(f"replay division J = {j:.3f}")
print(f"  +1 TP (a recovered division): score {0.1 * (division_j(tp + 1, fp, fn - 1) - j):+.5f}")
print(f"  +1 FP (a wrong fork)        : score {0.1 * (division_j(tp, fp + 1, fn) - j):+.5f}")
for label, jj in (("replay (in-sample)", j), ("hidden-set estimate", 0.40)):
    print(f"break-even precision of added forks, {label:>19}: {jj / (1 + jj):.0%}")
print("our add gate required >= 40 % precision on held-out scores")
# The public set is decided by roughly 30 annotated divisions: one of them moves the score by about 0.1 / 30
print(f"one division on a ~30-division public set: ~{0.1 / 30:.4f}")

## 2. Validation: the board cannot see a +0.001

**Everything on the training set is in-sample.** The public detector, Transformer and DeepCenter weights were
trained on all 199 training movies (their split manifest lists every movie), and the 4 visible "test" movies are
copies of training movies. We therefore never called a training-movie score out-of-fold. We used it only to
compare post-processing changes against each other on identical inputs. For anything upstream of
post-processing (detector, Transformer, association), a change also had to hold up on an external, never-trained-on
set (`val_biohub`: Zebrahub movies with automatic Ultrack tracks).

Here is the real progression. Every row is a scored submission, not a projection:

| System | Offline evidence | Public LB |
| --- | --- | ---: |
| Own classical baseline (DoG + LAP) | - | 0.616 |
| Own 3D U-Net detector | - | 0.669 |
| Public UNet + Transformer + ILP notebook (our control) | built-in validator | 0.936 |
| Public 0.947 pipeline, our run (association control) | - | 0.946 |
| Public "x138" notebook, as published | - | 0.953 |
| x138 + our own retrained V1284 coordinate head | leave-movie-out centre error -17 % | 0.952 |
| x138 + **b1c relink** | 4 visible movies 0.9180 -> 0.9290 | **0.954** |
| same, rule-based divisions **off** (measurement) | - | 0.914 |
| S1: delete forks whose continuing child barely moved | removes 7 FP forks, 0 TP in our tables | 0.950 |
| **S2: T3 add / remove** | replay: division J +0.020 / +0.093 (44b6 / 6bba) | **0.964** |
| S3: T3 x ranking model, add + remove | replay +0.0156 / +0.0016 | 0.958 |
| S2 variants (no deletion + reassignment, V5a, V5b) | replay predicts at most +1 division | 0.964 / 0.964 / 0.964 |
| S2 + J2 + V5a | replay +0.00353 | 0.964 |

Two things stand out. The biggest single gain came from a stage the ILP cannot express. And every version that
*deleted* forks lost on the board, even when our tables said the deleted forks were false.

In [ ]:
rows = [  # (label, Public LB, team contribution?)  - scored submissions
    ("Public UNet+Transformer+ILP (control)", 0.936, False),
    ("Public 0.947 pipeline", 0.946, False),
    ("Public x138 notebook", 0.953, False),
    ("+ b1c relink", 0.954, True),
    ("  rule-based divisions off (measurement)", 0.914, False),
    ("  S1: delete forks (geometry)", 0.950, True),
    ("+ T3 division recovery (S2)", 0.964, True),
    ("  S3: T3 x ranker, add + delete", 0.958, True),
    ("+ V5a + J2 smoothing", 0.964, True),
]
fig, ax = plt.subplots(figsize=(7.5, 3.6))
y = np.arange(len(rows))[::-1]
for yi, (label, lb, team) in zip(y, rows):
    ax.plot([0.91, lb], [yi, yi], color=GRID, lw=1, zorder=1)
    ax.scatter(lb, yi, s=60, color=BLUE if team else GREY, edgecolor="white", linewidth=1.5, zorder=3)
    ax.text(lb + 0.0012, yi, f"{lb:.3f}", va="center", color=INK, fontsize=9)
ax.set_yticks(y, [r[0] for r in rows])
ax.set_xlim(0.91, 0.972)
ax.set_xlabel("Public LB")
ax.scatter([], [], color=BLUE, label="team change"); ax.scatter([], [], color=GREY, label="public stack / control")
ax.legend(loc="lower left", frameon=False)
ax.set_title("Scored submissions (own-pipeline era, 0.616-0.669, off scale)", loc="left", color=INK, fontsize=11)
plt.tight_layout(); plt.show()

### Our experiment controls

After several attractive but unreal gains, every change had to pass the same gates:

- **Replay parity.** Inference runs once on a GPU and stores the ILP graph, the edge cache, the low-score detector
  peaks, the DeepCenter heatmaps and the duplicated-frame table of 88 annotated training movies. The replay then
  executes the *submitted notebook's own post-processing code* on CPU. The code is cut out of the notebook text at
  fixed anchors and never re-typed. Before a replay was trusted, its base configuration had to reproduce the
  submitted pipeline exactly: identical graphs entering the division stage and identical TP / FP / FN on 88 of 88
  movies. Final agreement was 0.00003.
- **Measurement submissions.** We spent one submission to switch rule-based divisions off (0.914). That told us
  what divisions are worth on the hidden embryos, which no training-movie number could.
- **Predict the board before reading it.** The replay said the S2 variants differ by at most one division. All of
  them came back at 0.964, as predicted.
- **Per embryo, and without the best movie.** A gain must be non-negative in both embryos and still positive after
  removing the single most helpful movie.
- **The real GPU run.** A change that should only move coordinates must reproduce on the 4 visible movies run end
  to end on Kaggle, with identical node sets.
- **Pre-registered promotion bar.** For example: +0.001 on the 88-movie replay, both embryos non-negative. A cheap
  screen may kill an idea; it cannot promote one.
- **Thresholds from within-embryo CV.** Division thresholds were chosen on movie-grouped CV scores *inside* an
  embryo and then reported on the other embryo with models that never saw it.

## 3. The pipeline that survived

```
3D + t movie
  -> dual-seed TemporalUNet3D, 8-view TTA            (public, pilkwang)
  -> V1284 coordinate head, |shift| < 2 um           (public, Anvith Pothula)
  -> association Transformer, forward + reverse time (public)
  -> ILP                                             (public)
  -> motion relink ............ + b1c jump-aware prior          <- team
  -> re-admission, gap closing, low-score gap filling (public, DeepCenter veto)
  -> rule-based safe division .. + T3 division recovery (V5a)   <- team
  -> short-track filter
  -> line-fit smoothing ........ J2 jump-aware                  <- team
  -> submission.csv
```

We kept the public stack exactly as published, except for two settings (the post-processing deadline and the edge
cache threshold). Every team change is a block inserted into the notebook at an anchor that must occur exactly
once. This keeps the upstream code untouched and makes each change removable.

### 3.1 b1c: a global-translation prior only where the motion model breaks

In 6bba, 55 of 66 audited movies contain **frozen transitions**, where frame t+1 is a bit-identical copy of frame
t (466 of them across 88 movies; none in 44b6). The upstream relink predicts each cell's next position from the
previous transition's flow field. Across a frozen transition that flow is exactly zero. On the following transition
the cells move two frames' worth, often together with a whole-embryo step. The true partner then falls outside the
5.5 um seed gate, and the Hungarian assignment links neighbours instead.

b1c classifies transitions as frozen / jump / normal. On **jump** transitions only, it adds a global translation T
to the prior. T comes from a point-set cross-correlation vote (the shift that makes most source-target pairs
coincide within 1.5 um), refined by mutual-nearest-neighbour ICP. Normal transitions are untouched, so on movies
without duplicated frames the output is bit-identical to the upstream relink.

On the 4 visible movies, errors on jump transitions fell from 31 missed / 49 false edges to 23 / 31, and the score
went from 0.9180 to 0.9290, all of it from the one movie with duplicated frames. On the public board the gain was
0.953 -> 0.954; a three-decimal score cannot show it more precisely.

The cell below reproduces the mechanism on a synthetic, drifting grid of cells.

In [ ]:
def vote_translation(src, tgt, radius=16.0, bandwidth=1.5):
    # Point-set cross-correlation: the displacement shared by the most source -> target pairs
    near = cKDTree(tgt).query_ball_point(src, r=radius)
    vec = np.array([tgt[j] - src[i] for i, js in enumerate(near) for j in js])
    tree = cKDTree(vec)
    counts = np.asarray(tree.query_ball_point(vec, r=bandwidth, return_length=True))
    best = vec[int(np.argmax(counts))]
    return np.median(vec[tree.query_ball_point(best, r=bandwidth)], axis=0)

def estimate_translation(src, tgt, iters=6):
    # Vote, then refine with the median residual of mutual nearest neighbours (translation-only ICP)
    shift = vote_translation(src, tgt)
    for _ in range(iters):
        _, j = cKDTree(tgt).query(src + shift)
        _, back = cKDTree(src + shift).query(tgt)
        mutual = back[j] == np.arange(len(src))
        new = np.median(tgt[j[mutual]] - src[mutual], axis=0)
        if np.allclose(new, shift, atol=1e-6):
            break
        shift = new
    return shift

def link(src, tgt, prior, gate=5.5):
    # Hungarian assignment of predicted positions to targets within the seed gate
    d = np.linalg.norm((src + prior)[:, None] - tgt[None], axis=2)
    cost = np.where(d <= gate, d, 1e6)
    r, c = linear_sum_assignment(cost)
    return {int(a): int(b) for a, b in zip(r, c) if cost[a, b] < 1e6}

rng = np.random.default_rng(5)
side, spacing, step = 4, 6.5, np.array([2.7, 3.6, 0.0])          # 64 cells 6.5 um apart, 4.5 um per frame
grid = np.stack(np.meshgrid(*[np.arange(side)] * 3, indexing="ij"), -1).reshape(-1, 3) * spacing
cells_t = grid + rng.normal(0, 1.0, grid.shape)
velocity = step + rng.normal(0, 0.4, grid.shape)
# frame t+1 is a duplicate of frame t (frozen), so frame t+2 is two real frames later
cells_t2 = cells_t + 2 * velocity + rng.normal(0, 0.2, grid.shape)
truth = np.arange(len(grid))

upstream_prior = np.zeros(3)          # the flow measured across the frozen transition is 0 um
T = estimate_translation(cells_t + upstream_prior, cells_t2)
for name, prior in (("upstream (zero flow after the frozen frame)", upstream_prior), ("b1c (prior + T)", T)):
    links = link(cells_t, cells_t2, prior)
    correct = sum(links.get(i) == i for i in truth)
    print(f"{name:<44} correct links {correct:>2}/{len(truth)}")
print("estimated T (um):", np.round(T, 2), " true mean step:", np.round(2 * velocity.mean(0), 2))

### 3.2 T3: learned division recovery after the rule-based stage

**Why a post-ILP stage.** The ILP's costs are edge -p, appearance 0, disappearance 2 and division 1.2. Adding a
second daughter changes the objective by `-p + 1.2 > 0` for any probability, so the ILP never forks. On the public
x138 output no node had two children. All divisions came from a rule-based "safe division" stage (distance,
symmetry, mutual nearest neighbour, divergence at t+2, a DeepCenter veto and per-frame budgets), and it missed many.

**The hook.** Right after safe division, for each movie:

1. **Candidates.** Every node P with exactly one child D1, and every track start D2 at t+1 with P->D1 <= 12 um,
   P->D2 <= 12 um and D1-D2 <= 16 um. This is wider than the rules; no ground truth is involved.
2. **A model-free gate for additions (`prob_angle_t2`).** The association Transformer's probability p(P->D2) must
   have been cached (> 0.48, so the pair was a real candidate edge), the angle D1-P-D2 must be >= 90 deg, and the
   two sisters must move further apart at t+2.
3. **T3 scores only the parents that matter:** gated additions plus every rule-based fork (114, 51, 19 and 170
   parents on the 4 visible movies).
4. **Decisions.** Add the fork (P -> the D2 closest to the mirror image of D1) when T3 >= 0.3. Delete a rule-based
   fork only when T3 < 0.01. Never create a node, and never put two forks on one division (+-1 frame, 10 um). The
   learned additions are exempt from the rule stage's per-frame budget (V5a).

**The model.** T3 is a small 3D CNN on a 3-channel crop (frames t-1, t, t+1) of 16 x 48 x 48 native voxels (26 x
19.5 x 19.5 um) centred on P. A dividing mother is condensed and bright at t, and two nuclei appear at t+1.
Positives are the annotated divider at t, t-1 and t+1 (the metric accepts a fork one frame early or late),
oversampled 10x, with xy flips, 90-degree rotations, a 1-slice / 8-pixel shift and an intensity scale. Negatives are
annotated cells with one child, away from any division. The submission averages 3 seeds trained on all 199 movies
over 2 test-time views.

| Evaluation | Result |
| --- | --- |
| Embryo holdout 44b6 -> 6bba | AUC 0.864, AP 0.554 |
| Embryo holdout 6bba -> 44b6 | AUC 0.942, AP 0.694 |
| Precision of the added forks, 88-movie replay | 18 / 28 = 64 % |
| Division J change of the additions, replay | +0.020 (44b6), +0.093 (6bba) |
| Public LB | 0.954 -> **0.964** |

**Thresholds.** `tau_add` is the lowest threshold whose added forks reach 40 % precision (break-even is 19-29 %,
see section 1) on within-embryo movie-CV scores. `tau_del` stays just below the largest value that deletes no true
fork. That the cross-embryo check matters was proven by a failure with the combined score we tried in S3: a
deletion threshold chosen on 6bba (0.92) would have removed true forks on 44b6, while one chosen on 44b6 was safe
both ways.

In [ ]:
try:
    import torch.nn as nn
    import torch

    def block(cin, cout, k):
        return nn.Sequential(nn.Conv3d(cin, cout, k, padding=tuple(v // 2 for v in k), bias=False),
                             nn.BatchNorm3d(cout), nn.SiLU())

    w = 16
    t3 = nn.Sequential(   # z voxels are 4x larger than xy voxels: pool xy first
        block(3, w, (1, 3, 3)), block(w, w, (3, 3, 3)), nn.MaxPool3d((1, 2, 2)),
        block(w, 2 * w, (3, 3, 3)), block(2 * w, 2 * w, (3, 3, 3)), nn.MaxPool3d(2),
        block(2 * w, 4 * w, (3, 3, 3)), block(4 * w, 4 * w, (3, 3, 3)), nn.MaxPool3d(2),
        block(4 * w, 8 * w, (3, 3, 3)), nn.AdaptiveAvgPool3d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(8 * w, 1))
    x = torch.zeros(2, 3, 16, 48, 48)      # (batch, frames t-1/t/t+1, z, y, x)
    print("T3 parameters:", sum(p.numel() for p in t3.parameters()), " output:", tuple(t3(x).shape))
except ImportError:
    print("torch is not installed; the T3 definition is in src/biohub_tracking/division/t3.py")

### 3.3 J2: smoothing that knows where the frames jump

The last upstream stage moves every node 80 % of the way toward a straight line fitted through its track +-2 frames.
Across a frozen transition or an embryo jump, that line mixes two alignments and drags nodes on both sides toward
the step. J2 stops the walk at frozen transitions and subtracts the translation b1c applied at a jump, so the fit
sees the track as if the embryo had not moved. Without special transitions it is bit-identical to the upstream
smoothing, and if it ever raises, the insured version falls back to it.

On the 88-movie replay J2 added +0.00301 (with V5a +0.00353: 44b6 +0.00316, 6bba +0.00374, edges +54 TP / -116 FP /
-54 FN). The public score stayed 0.964 at three decimals.

In [ ]:
def smooth(y, weight=0.8, window=2, cut=(), shift=None):
    # Line-fit smoothing along one track. cut: transitions t -> t+1 where the walk stops;
    # shift: {t: jump at transition t}, subtracted so the walk sees an unmoved embryo
    shift = shift or {}
    out = y.copy()
    for i in range(len(y)):
        pts = [(0, y[i])]
        off = 0.0
        for s in range(1, window + 1):                       # backward
            t = i - s
            if t < 0 or t in cut:
                break
            off -= shift.get(t, 0.0)
            pts.append((-s, y[t] - off))
        off = 0.0
        for s in range(1, window + 1):                       # forward
            t = i + s - 1
            if i + s >= len(y) or t in cut:
                break
            off += shift.get(t, 0.0)
            pts.append((s, y[i + s] - off))
        if len(pts) >= 3:
            dt, v = np.array(pts).T
            out[i] = (1 - weight) * y[i] + weight * np.polyval(np.polyfit(dt, v, 1), 0.0)
    return out

frames = np.arange(12)
rng = np.random.default_rng(0)
y = 0.6 * frames + np.where(frames >= 6, 6.0, 0.0) + rng.normal(0, 0.15, frames.size)  # +6 um embryo step at 5 -> 6
upstream = smooth(y)
j2 = smooth(y, shift={5: 6.0})                                  # the translation b1c applied at transition 5

fig, ax = plt.subplots(figsize=(7.5, 3.0))
ax.plot(frames, y, "o", color=GREY, label="detected position")
ax.plot(frames, upstream, "-", color=MUTED, lw=1.5, label="upstream smoothing")
ax.plot(frames, j2, "-", color=BLUE, lw=2, label="J2")
ax.axvspan(5, 6, color=GRID, alpha=0.6, lw=0)
ax.text(5.5, y.max(), "jump", ha="center", va="top", color=MUTED)
ax.set_xlabel("frame"); ax.set_ylabel("y (um)")
ax.legend(frameon=False, loc="upper left")
ax.set_title("One track across an embryo jump", loc="left", color=INK, fontsize=11)
plt.tight_layout(); plt.show()
print("mean |move| of the 4 nodes next to the jump: upstream",
      np.round(np.abs(upstream - y)[4:8].mean(), 2), "um, J2", np.round(np.abs(j2 - y)[4:8].mean(), 2), "um")

### 3.4 Engineering that made 65 submissions safe

- **Never hand-edit third-party code.** Every change is an anchor-checked insertion. The edits are invertible, so
  each final-day candidate is regenerated from one core notebook and verified against the SHA-256 of the code that
  was actually submitted.
- **Fail loudly.** A hidden-test rerun takes an estimated 25,000-28,000 s. The upstream code silently switched off
  its repair stages for the remaining movies once a 27,000 s deadline passed. We raised the deadline to 36,000 s and
  made a degraded run *fail* instead of quietly submitting a different pipeline.
- **Audit what you submit.** An independent audit of the 0.964 notebook found no fatal bug, but it did find one
  silent risk: a movie whose post-processing raised an exception would be written from the bare ILP graph without
  any warning. The blocks we added afterwards (J2, the frozen-frame consensus) each fall back to their own input on
  an exception instead of taking the whole movie down.

## 4. What did not work, and what each failure taught us

### 4.1 Our own detector: stop earlier than it feels natural

A classical pipeline scored 0.616 and our own sparse-point 3D U-Net reached 0.669. Calibration and distillation
variants collapsed (0.366, 0.373). The public pretrained stack scored 0.936 on the first comparison. We stopped
the own-detector line and spent the remaining time where the strong stack was weak.

### 4.2 Better association models did not help, because association was not the problem

| Association variant (vs 0.946 control) | Public LB |
| --- | ---: |
| hard-negative margin | 0.945 |
| source-target relative-rank prior | 0.945 |
| soft mutual-best rank support | 0.944 |
| FOCUS-3D primary linker | 0.941 |
| SuperGlue-style dense matching | 0.870 |
| Trackastra-style 5-frame association | 0.946 |
| MeMOTR-style memory association | 0.946 |

In an earlier six-arm experiment every arm had `wrong_association = 0`. A stage-by-stage attribution of every missed
edge on 72 movies then showed that 45 % of lost edges were missing from the candidate pairs handed to the ILP, 18 %
were broken by the motion relink (the stage we later found failing on duplicated frames), and none was explained by
a low Transformer score on an existing candidate. **Lesson:** measure where the loss is before choosing what to model.

### 4.3 A detector fine-tune that looked promising internally

A local-softmax fine-tune of the public weights lost 0.046 on our internal diagnostic *and* 0.046 on the external
`val_biohub` set, so it was stopped before costing a submission. This is exactly the case the external set exists
for.

### 4.4 Deleting forks: the labels cannot see the unannotated majority

S1 deleted rule-based forks whose continuing child barely moved; our tables said it removed 7 false forks and no true
ones. The public score fell from 0.954 to 0.950. S3 added and deleted with a stronger score and fell to 0.958.
Because most forks sit on unannotated cells, "no annotated true fork removed" does not mean "no real division
removed". **Lesson:** under sparse labels, additions are measurable and deletions are not; delete only at extreme
confidence (T3 < 0.01).

### 4.5 In-sample gains that did not transfer

Weaker smoothing (weight 0.6 instead of 0.8) gained +0.0035 on the replay. Three things were wrong with it. On 44b6
the gain came almost entirely from one movie (+0.0004 without it). The real GPU run on the visible movies got
*worse*. And the detector coordinates it relied on are in-sample, so they are more accurate than on hidden embryos,
which favours less smoothing. Rejected. J2, the principled version of the same idea, was positive in both embryos
and on the GPU run.

### 4.6 The post-processing knobs were saturated

Single-knob changes to gap filling (best +0.00010), re-admission (-0.00004 to +0.00003), four rule-based node
relocalisation methods (none better than the baseline) and a low-probability second-daughter branch (its gain was
not reproduced across folds and rested on three events) all failed the +0.001 bar. **Lesson:** once the replay says
the remaining errors are in detection and localisation (section 5), stop tuning post-processing.

## 5. Where the remaining error is

The final audit attributed each of the 2,810 GT edges still missed by S2 on the 88-movie replay (exclusive
categories, first match wins):

In [ ]:
causes = [  # (cause, share of missed GT edges, fixable by linking / post-processing?)
    ("Node localisation (> 3.5 um off, or pair >= 7 um)", 46.2, False),
    ("Detection kept out by the ILP", 18.3, False),
    ("Removed by the short-track filter", 12.7, False),
    ("No detection within 7 um", 11.4, False),
    ("Wrong partner, correct pair not a candidate", 4.3, False),
    ("Only below-threshold detector peaks", 3.5, False),
    ("Nearby prediction taken by another GT node", 2.2, False),
    ("Wrong partner, correct pair was a candidate", 1.5, True),
]
fig, ax = plt.subplots(figsize=(7.5, 3.2))
y = np.arange(len(causes))[::-1]
ax.barh(y, [c[1] for c in causes], color=[BLUE if c[2] else GREY for c in causes], height=0.62)
for yi, (_, share, _) in zip(y, causes):
    ax.text(share + 0.6, yi, f"{share:.1f} %", va="center", color=INK, fontsize=9)
ax.set_yticks(y, [c[0] for c in causes]); ax.set_xlim(0, 55); ax.set_xlabel("share of missed GT edges (%)")
ax.set_title("S2 residual misses, 88 movies (blue: fixable by linking alone)", loc="left", color=INK, fontsize=11)
plt.tight_layout(); plt.show()

| Scope | Movies | Edge TP / FP / FN | Adjusted edge J | Division TP / FP / FN | Score |
| --- | ---: | --- | ---: | --- | ---: |
| 44b6 | 22 | 6,463 / 406 / 318 | 0.9145 | 6 / 10 / 20 | 0.9312 |
| 6bba | 66 | 57,064 / 2,459 / 2,492 | 0.9199 | 38 / 25 / 87 | 0.9452 |
| All | 88 | 63,527 / 2,865 / 2,810 | 0.9193 | 44 / 35 / 107 | 0.9430 |

- **Linking is capped.** Perfect choices among existing candidates would add at most +0.0006, and perfect candidate
  generation +0.0018. Two thirds of the loss needs a better detector or coordinate model.
- **Division recall is the weak point** (107 misses vs 44 hits). Of 120 missed edges caused by divisions, 80 are
  localisation errors of the daughters.
- **Embryos differ.** 44b6 scores lower and has no duplicated frames; every frozen-frame fix acts only on 6bba.
  The DeepCenter prior was trained on 44b6, and 25 % of the forks decided by its 0.25 veto lie within +-0.05 of the
  threshold, a known sensitivity on hidden embryos.
- **Node count cuts both ways.** Lowering the minimum track length from 6 to 5 recovers +43 true edges but adds
  20,000 nodes, a net loss (-0.00011). That is why the filter stayed as it was.

## 6. Contribution ledger

| Idea | Measurement | Decision |
| --- | --- | --- |
| Adopt the strongest public stack, stop own detector | own 0.669 vs public 0.936 -> 0.953 | keep (credit upstream) |
| Error attribution before modelling | 45 % of lost edges missing from candidates; 63 % of final misses detection-side | keep as a standing practice |
| b1c jump-aware relink | visible 4: 0.9180 -> 0.9290; Public 0.953 -> 0.954 | keep |
| Divisions-off measurement submission | 0.954 -> 0.914: divisions worth 0.040 | keep (a submission well spent) |
| T3 learned division recovery (S2) | add precision 64 %; Public 0.954 -> 0.964 | keep |
| V5a: learned adds exempt from the rule budget | replay +1 division; Public 0.964 | keep |
| J2 jump-aware smoothing | replay +0.00353, both embryos positive | keep |
| Exact CPU replay of the submitted code | parity 0.00003; predicted the S2-variant ties | keep |
| Own V1284 head (30 movies) | centre error -17 %, Public 0.952 < 0.953 | reject (public head kept) |
| Seven association variants | 0.870-0.946 vs 0.946 control | reject |
| Local-softmax detector fine-tune | -0.046 internal and external | reject |
| Fork deletion (S1, S3) | Public 0.950 / 0.958 | reject: labels cannot see deletions |
| Weaker smoothing (0.6) | replay +0.0035, GPU run worse, one-movie gain | reject: in-sample |
| Gap filling / re-admission / relocalisation / low-p branch | all < +0.001 | reject: saturated |
| Frozen-frame coordinate consensus | replay +0.00103 (6bba +0.00115), +0.00087 without the best movie | final-day candidate |
| F03 fine-tuned coordinate head | failed its pre-registered gate; visible 4 +0.00135 | final-day hedge only |

## 7. Where we draw the line

Every component in the final notebook has a physical job. b1c repairs the one place the motion model is known to
break. T3 supplies the divisions the optimiser cannot express. J2 keeps the smoothing from mixing two alignments.
We allowed a change when it was positive on the replay in both embryos, survived removing its best movie, and
reproduced on the real GPU run. We did not allow a change because the board liked it, because an oracle showed
headroom, or because in-sample coordinates made it look good.

On the final day we submitted small additions on top of the confirmed 0.964 core. The frozen-frame coordinate
consensus averages the two copies of a nucleus across a duplicated frame; it passed every pre-registered check. The
F03 coordinate head failed its gate and was submitted only as a hedge. A teammate's variant recovers missed nuclei
with an extra high-resolution detection head (3 um de-duplication, at most +5 % nodes). Final selection and private
results: **TBD** (to be filled in after the deadline).

## 8. Reproducibility checklist

- The submitted notebook is published with its code cells byte-identical to the Kaggle kernel; each final-day
  candidate is regenerated from it and checked against the SHA-256 of the submitted code.
- The team's components are also a tested Python package (`src/biohub_tracking`). Parity tests execute the code
  embedded in the notebook on random synthetic movies and require identical output from the package.
- Local scores use only the organisers' `tracking_cellmot` implementation, on predictions rounded through the
  submission CSV schema, reported per embryo.
- Every training-movie number is labelled in-sample; the external set was never used for gradients or tuning.
- T3 thresholds come from within-embryo movie CV and are reported on the other embryo.
- Seeds, crop sizes, test-time views and thresholds are fixed in code; the F03 head is checked by SHA-256 at load,
  and every run writes a manifest of its resolved configuration.
- Negative results are kept with their controls. A failed configuration is not a claim that the whole idea is
  impossible.

## Closing thought

This competition looks like a tracking-model contest, but the decisive question was simpler: *where does the
score go, and can the pipeline even express the fix?* The ILP could not express a division. The motion prior
could not survive a duplicated frame. The smoothing could not see a jump. Fixing those three blind spots moved
the score more than every new model we trained.

## Acknowledgements

This solution stands on public work shared during the competition, used under its licenses (Apache-2.0 notebooks,
CC0 model datasets):

- **Anvith Pothula**: the "biohub x138" notebook and the V1284 coordinate-refinement head (our base notebook).
- **Teddy Tennant**: the "frontier947" notebooks (flow-prior relink, re-admission, gap filling).
- **Reyhan Ksatria**: the 0.947 post-processing pipeline (motion relink, gap closing, safe division, DeepCenter veto).
- **pilkwang**: the pretrained TemporalUNet3D detectors, association Transformer and DeepCenter model.
- **The organisers (royerlab)**: the data, the official metric (`tracking_cellmot`) and the baseline.